# BCI Calibration Data Analysis
This notebook analyzes the calibration CSV files to compare 'concentrate' and 'relax' states.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy import stats
from sklearn.metrics import roc_curve, auc

# Set plot styles
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading & Preprocessing
Loads all `calibration_*.csv` files and combines them into a single DataFrame.

In [ ]:
all_files = glob.glob("calibration_*.csv")
if not all_files:
    print("No calibration files found! Please run the dashboard and record some data first.")
else:
    df_list = []
    for file in all_files:
        temp_df = pd.read_csv(file)
        df_list.append(temp_df)
    
    df = pd.concat(df_list, ignore_index=True)
    print(f"Loaded {len(df)} rows from {len(all_files)} files.")
    print("Available Labels:", df['label'].unique())
    
    # You can filter specific labels if needed
    # df = df[df['label'].isin(['concentrate', 'relax'])]
    display(df.head())

## 2. Descriptive Statistics
Calculates the mean, standard deviation, and variance for each metric grouped by the brain state label.

In [ ]:
# Define metrics to analyze (you can add more columns here)
metrics_to_analyze = ['avg_beta_alpha', 'avg_beta_theta_alpha', 'avg_one_over_alpha']

# Optionally add channel-specific metrics
for ch in range(1, 9):
    metrics_to_analyze.append(f"ch{ch}_beta_alpha")
    
if 'df' in locals():
    # Group by label and calculate statistics
    desc_stats = df.groupby('label')[metrics_to_analyze].agg(['mean', 'std', 'var']).T
    display(desc_stats)

## 3. Visualizations
Boxplots and Violin plots to visually compare the distribution of the attention indexes between states.

In [ ]:
if 'df' in locals():
    # Change this to any metric you want to visualize
    key_metric = 'avg_beta_alpha'

    plt.figure(figsize=(10, 6))
    sns.boxplot(x='label', y=key_metric, data=df, palette='Set2')
    sns.stripplot(x='label', y=key_metric, data=df, color='black', alpha=0.3, jitter=True)
    plt.title(f'Distribution of {key_metric} by Brain State')
    plt.ylabel(key_metric)
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.violinplot(x='label', y=key_metric, data=df, palette='Set2', inner='quartile')
    plt.title(f'Violin Plot of {key_metric} by Brain State')
    plt.ylabel(key_metric)
    plt.show()

## 4. Statistical Comparison (ANOVA)
Performs a One-way ANOVA to determine if the difference between the states is statistically significant (p < 0.05).

In [ ]:
if 'df' in locals():
    labels = df['label'].unique()
    if len(labels) >= 2:
        # We will compare the first two labels found
        group1_label = labels[0]
        group2_label = labels[1]
        
        group1 = df[df['label'] == group1_label]
        group2 = df[df['label'] == group2_label]
        
        print(f"ANOVA Comparison: '{group1_label}' vs '{group2_label}'\n")
        
        results = []
        for metric in metrics_to_analyze:
            f_stat, p_val = stats.f_oneway(group1[metric].dropna(), group2[metric].dropna())
            significant = "Yes" if p_val < 0.05 else "No"
            results.append({
                'Metric': metric,
                'F-Statistic': f_stat,
                'p-value': p_val,
                'Significant': significant
            })
            
        anova_df = pd.DataFrame(results).sort_values(by='p-value')
        display(anova_df)
    else:
        print("Need at least 2 different labels in the dataset to perform ANOVA.")

## 5. Threshold Evaluation & Criterion
Uses ROC Curve and AUC to evaluate how well the metric separates the two classes, and finds the optimal threshold using Youden's J statistic.

In [ ]:
if 'df' in locals() and len(labels) >= 2:
    # Set which label is the "positive" target class (e.g., 'concentrate')
    # Change target_label to match your specific concentrate label string
    target_label = labels[0] 
    
    # Create binary target array
    y_true = (df['label'] == target_label).astype(int)
    y_scores = df[key_metric]
    
    # Calculate ROC metrics
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    # Find Optimal Threshold using Youden's J statistic (J = TPR - FPR)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 1. Plot ROC Curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.scatter(fpr[optimal_idx], tpr[optimal_idx], marker='o', color='red', s=100, zorder=5,
                label=f'Optimal Threshold = {optimal_threshold:.4f}')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic ({key_metric})')
    plt.legend(loc="lower right")
    plt.show()
    
    print(f"Optimal Threshold for {key_metric} to detect '{target_label}': {optimal_threshold:.4f}")
    
    # 2. Plot Overlapping Distributions (KDE) with Threshold Line
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df, x=key_metric, hue='label', fill=True, common_norm=False, palette='Set1', alpha=0.5)
    plt.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold ({optimal_threshold:.4f})')
    plt.title(f'KDE Distribution of {key_metric} with Optimal Threshold')
    plt.legend()
    plt.show()